# SentryMesh Guardian: Rigorous Evaluation & Error Analysis

This notebook runs the held-out evaluation on `data/processed/test.csv` (15% untouched split).

### Diagnostic Analysis Performed:
1. **Per-Class Metrics**: Precision, Recall, F1-Score, and Support.
2. **Macro and Weighted F1-scores**.
3. **Confusion Matrix Heatmap**.
4. **False Positive Audit**: Inspecting legitimate messages (e.g. bank OTP alerts, scholarship reminders) to guarantee zero shortcut learning.
5. **False Negative Audit**: Unpacking scam vectors that evade classification.
6. **Slice Evaluation**: Channel slice (`sms`, `call`, `email`, `qr`), Language slice (`en`, `hi-en`, `kn-en`), and Data Source slice.

In [ ]:
!pip install -q pandas scikit-learn matplotlib seaborn
import sys
sys.path.append("..")
from scripts.evaluate_model import run_evaluation

results = run_evaluation()
print("Evaluation completed successfully:", results)

## 1. Inspect Confusion Matrix

In [ ]:
import json
import pandas as pd

with open("../reports/confusion_matrix.json", "r") as f:
    cm = json.load(f)

df_cm = pd.DataFrame(cm).fillna(0).astype(int)
df_cm

## 2. Inspect Slice Evaluation

In [ ]:
with open("../reports/slice_metrics.json", "r") as f:
    slices = json.load(f)

print("--- Channel Slice Accuracy ---")
for k, v in slices["channel_slice"].items():
    print(f"  {k:12s}: Acc={v['accuracy']:.4f} (n={v['samples']})")

print("\n--- Language Slice Accuracy ---")
for k, v in slices["language_slice"].items():
    print(f"  {k:12s}: Acc={v['accuracy']:.4f} (n={v['samples']})")

## 3. False Positive & False Negative Audits

In [ ]:
fp_df = pd.read_csv("../reports/false_positives.csv")
fn_df = pd.read_csv("../reports/false_negatives.csv")

print(f"Total False Positives (Legitimate misclassified): {len(fp_df)}")
print(f"Total False Negatives (Scams missed): {len(fn_df)}")
if len(fp_df) > 0:
    print("\nFP Samples:", fp_df.head(3))
if len(fn_df) > 0:
    print("\nFN Samples:", fn_df.head(3))